# 🚨 Dashboard: Incautación de Estupefacientes en Colombia (2010–2025)

**Dataset:** `Incautación_de_Estupefacientes__20260423.csv`  
**Fuente:** Datos abiertos Colombia  
**Registros:** ~1.95 millones de eventos  

---

Este notebook construye un pipeline de análisis completo:
1. Carga y limpieza
2. KPIs resumen
3. Tendencia temporal interactiva
4. Mapa de calor por departamento × año
5. Treemap jerárquico departamento → sustancia
6. Sunburst sustancia → departamento
7. Comparativo de crecimiento por sustancia (índice base 2010)
8. Top municipios por tipo de droga
9. Heatmap mensual de eventos
10. Cierre analítico


## Parte 1 · Carga y limpieza

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_PATH = Path('Incautación_de_Estupefacientes__20260423.csv')

df = pd.read_csv(DATA_PATH, low_memory=False)
df = df.rename(columns={
    'DEPARTAMENTO': 'departamento',
    'MUNICIPIO':    'municipio',
    'CODIGO DANE':  'codigo_dane',
    'CLASE BIEN':   'sustancia',
    'FECHA HECHO':  'fecha',
    'CANTIDAD':     'cantidad',
})

df['fecha']    = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce')
df = df.dropna(subset=['fecha', 'cantidad']).sort_values('fecha')

df['anio']    = df['fecha'].dt.year
df['mes_num'] = df['fecha'].dt.month
df['mes_nom'] = df['fecha'].dt.strftime('%b')
df['periodo'] = df['fecha'].dt.to_period('M').astype(str)

# Excluir 2026 (año incompleto)
df = df[df['anio'] <= 2025]

print(f'Registros cargados : {len(df):,}')
print(f'Sustancias          : {", ".join(df["sustancia"].unique())}')
print(f'Departamentos       : {df["departamento"].nunique()}')
print(f'Municipios          : {df["municipio"].nunique()}')
print(f'Rango temporal      : {df["anio"].min()} – {df["anio"].max()}')
df.head()


## Parte 2 · KPIs generales

Primero una vista de resumen de alto nivel: total incautado, número de eventos, sustancia dominante y departamento más activo.

In [ ]:
total_kg    = df['cantidad'].sum() / 1_000  # gramos → kilogramos aprox
total_ev    = len(df)
top_sust    = df.groupby('sustancia')['cantidad'].sum().idxmax()
top_depto   = df.groupby('departamento')['cantidad'].sum().idxmax()
anio_pico   = df.groupby('anio')['cantidad'].sum().idxmax()

fig = go.Figure()

kpis = [
    ('Total incautado (millones)', f'{df["cantidad"].sum()/1e6:,.1f} M', '#e74c3c'),
    ('Eventos registrados',        f'{total_ev:,}',                       '#2980b9'),
    ('Sustancia predominante',     top_sust,                               '#8e44ad'),
    ('Depto. más activo',          top_depto,                              '#27ae60'),
    ('Año pico de incautaciones',  str(anio_pico),                         '#d35400'),
]

for i, (label, value, color) in enumerate(kpis):
    fig.add_trace(go.Indicator(
        mode='number',
        value=None,
        title={'text': f'<b>{value}</b><br><span style="font-size:13px;color:gray">{label}</span>'},
        domain={'row': 0, 'column': i},
    ))

fig.update_layout(
    grid={'rows': 1, 'columns': 5},
    height=160,
    margin=dict(t=20, b=10, l=10, r=10),
    paper_bgcolor='white',
    title_text='📊 KPIs — Incautación de Estupefacientes Colombia 2010–2025',
    title_font_size=16,
)
fig.show()


## Parte 3 · Tendencia temporal por sustancia

Línea interactiva con cantidad total incautada por año y sustancia. Puedes hacer clic en la leyenda para mostrar/ocultar sustancias.

In [ ]:
trend = (
    df.groupby(['anio', 'sustancia'], as_index=False)['cantidad']
    .sum()
)
trend['cantidad_m'] = trend['cantidad'] / 1e6

fig = px.line(
    trend,
    x='anio',
    y='cantidad_m',
    color='sustancia',
    markers=True,
    title='Cantidad total incautada por año y sustancia (millones de unidades)',
    labels={'anio': 'Año', 'cantidad_m': 'Cantidad (millones)', 'sustancia': 'Sustancia'},
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_traces(line_width=2.5, marker_size=7)
fig.update_layout(
    hovermode='x unified',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis=dict(tickmode='linear', dtick=1),
)
fig.show()


## Parte 4 · Área apilada: composición anual

Muestra cómo cambia la mezcla de sustancias incautadas año a año.

In [ ]:
fig = px.area(
    trend,
    x='anio',
    y='cantidad_m',
    color='sustancia',
    title='Composición anual de incautaciones (área apilada)',
    labels={'anio': 'Año', 'cantidad_m': 'Cantidad (millones)', 'sustancia': 'Sustancia'},
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_layout(
    hovermode='x unified',
    height=420,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis=dict(tickmode='linear', dtick=1),
)
fig.show()


## Parte 5 · Mapa de calor: departamento × año

Identifica qué departamentos concentran más incautaciones y en qué periodos.

In [ ]:
top20_deptos = (
    df.groupby('departamento')['cantidad']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .index.tolist()
)

heat = (
    df[df['departamento'].isin(top20_deptos)]
    .groupby(['departamento', 'anio'], as_index=False)['cantidad']
    .sum()
)
heat['cantidad_m'] = heat['cantidad'] / 1e6

pivot = heat.pivot(index='departamento', columns='anio', values='cantidad_m').fillna(0)

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.astype(str).tolist(),
    y=pivot.index.tolist(),
    colorscale='YlOrRd',
    colorbar_title='Millones',
    hoverongaps=False,
    hovertemplate='<b>%{y}</b><br>Año: %{x}<br>Cantidad: %{z:.2f}M<extra></extra>',
))
fig.update_layout(
    title='Mapa de calor — Cantidad incautada por departamento y año (top 20 deptos)',
    xaxis_title='Año',
    yaxis_title='Departamento',
    height=600,
    margin=dict(l=160),
)
fig.show()


## Parte 6 · Treemap jerárquico: departamento → sustancia

Visualización proporcional de quién incauta qué. El tamaño del rectángulo es proporcional a la cantidad total.

In [ ]:
tree = (
    df[df['departamento'].isin(top20_deptos)]
    .groupby(['departamento', 'sustancia'], as_index=False)['cantidad']
    .sum()
)
tree['cantidad_m'] = tree['cantidad'] / 1e6

fig = px.treemap(
    tree,
    path=[px.Constant('Colombia'), 'departamento', 'sustancia'],
    values='cantidad_m',
    color='cantidad_m',
    color_continuous_scale='Reds',
    title='Treemap — Distribución jerárquica: País → Departamento → Sustancia',
    labels={'cantidad_m': 'Cantidad (M)'},
)
fig.update_traces(
    textinfo='label+percent parent',
    hovertemplate='<b>%{label}</b><br>Cantidad: %{value:.2f}M<br>% del padre: %{percentParent:.1%}<extra></extra>',
)
fig.update_layout(height=560, margin=dict(t=50, l=10, r=10, b=10))
fig.show()


## Parte 7 · Sunburst: sustancia → departamento

Perspectiva inversa: qué departamentos dominan dentro de cada sustancia.

In [ ]:
sun = (
    df[df['departamento'].isin(top20_deptos)]
    .groupby(['sustancia', 'departamento'], as_index=False)['cantidad']
    .sum()
)
sun['cantidad_m'] = sun['cantidad'] / 1e6

fig = px.sunburst(
    sun,
    path=['sustancia', 'departamento'],
    values='cantidad_m',
    color='sustancia',
    title='Sunburst — Sustancia → Departamento (haz clic para explorar)',
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_traces(
    hovertemplate='<b>%{label}</b><br>Cantidad: %{value:.2f}M<br>% del total: %{percentRoot:.1%}<extra></extra>',
)
fig.update_layout(height=560, margin=dict(t=50, l=10, r=10, b=10))
fig.show()


## Parte 8 · Índice de crecimiento (base 2010 = 100)

¿Qué sustancia ha crecido más desde 2010? Normalizamos a 100 para comparar en la misma escala.

In [ ]:
base = trend[trend['anio'] == 2010][['sustancia', 'cantidad']].set_index('sustancia')['cantidad']

indice = trend.copy()
indice['indice'] = indice.apply(lambda r: r['cantidad'] / base.get(r['sustancia'], 1) * 100, axis=1)

fig = px.line(
    indice,
    x='anio',
    y='indice',
    color='sustancia',
    markers=True,
    title='Índice de crecimiento por sustancia (base 2010 = 100)',
    labels={'anio': 'Año', 'indice': 'Índice (2010=100)', 'sustancia': 'Sustancia'},
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.add_hline(y=100, line_dash='dash', line_color='gray', annotation_text='Base 2010')
fig.update_layout(
    hovermode='x unified',
    height=430,
    xaxis=dict(tickmode='linear', dtick=1),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()


## Parte 9 · Top 15 municipios por sustancia

Barras agrupadas interactivas. Usa el menú desplegable de Plotly para filtrar por sustancia.

In [ ]:
sustancias = df['sustancia'].unique().tolist()

fig = go.Figure()

for sust in sustancias:
    muni_df = (
        df[df['sustancia'] == sust]
        .groupby('municipio', as_index=False)['cantidad']
        .sum()
        .sort_values('cantidad', ascending=False)
        .head(15)
    )
    fig.add_trace(go.Bar(
        name=sust,
        x=muni_df['municipio'],
        y=muni_df['cantidad'] / 1e6,
        visible=(sust == sustancias[0]),
        marker_color=px.colors.qualitative.Bold[sustancias.index(sust) % 10],
        hovertemplate='<b>%{x}</b><br>Cantidad: %{y:.2f}M<extra></extra>',
    ))

buttons = []
for i, sust in enumerate(sustancias):
    visibility = [j == i for j in range(len(sustancias))]
    buttons.append(dict(
        label=sust,
        method='update',
        args=[{'visible': visibility},
              {'title': f'Top 15 municipios — {sust}'}],
    ))

fig.update_layout(
    title=f'Top 15 municipios — {sustancias[0]}',
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.0, y=1.15,
        showactive=True,
        buttons=buttons,
    )],
    xaxis_tickangle=-40,
    yaxis_title='Cantidad (millones)',
    xaxis_title='Municipio',
    height=480,
    showlegend=False,
)
fig.show()


## Parte 10 · Heatmap mensual de eventos

¿Hay estacionalidad en los operativos de incautación? Este heatmap muestra el número de eventos por mes y año.

In [ ]:
MESES = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

evt = (
    df.groupby(['anio', 'mes_num'])
    .size()
    .reset_index(name='eventos')
)
evt['mes_nom'] = evt['mes_num'].apply(lambda m: MESES[m-1])

pivot_evt = evt.pivot(index='anio', columns='mes_nom', values='eventos')
pivot_evt = pivot_evt[MESES]  # orden correcto

fig = go.Figure(go.Heatmap(
    z=pivot_evt.values,
    x=MESES,
    y=pivot_evt.index.astype(str).tolist(),
    colorscale='Blues',
    colorbar_title='Eventos',
    hovertemplate='<b>%{y} – %{x}</b><br>Eventos: %{z:,}<extra></extra>',
))
fig.update_layout(
    title='Heatmap mensual — Número de eventos de incautación por año y mes',
    xaxis_title='Mes',
    yaxis_title='Año',
    height=460,
)
fig.show()


## Parte 11 · Evolución por departamento — animación temporal

Gráfico de barras animado año a año para los top 10 departamentos.

In [ ]:
top10 = (
    df.groupby('departamento')['cantidad']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

anim_df = (
    df[df['departamento'].isin(top10)]
    .groupby(['anio', 'departamento'], as_index=False)['cantidad']
    .sum()
)
anim_df['cantidad_m'] = anim_df['cantidad'] / 1e6
anim_df['anio_str'] = anim_df['anio'].astype(str)

fig = px.bar(
    anim_df,
    x='cantidad_m',
    y='departamento',
    orientation='h',
    animation_frame='anio_str',
    color='departamento',
    range_x=[0, anim_df['cantidad_m'].max() * 1.1],
    title='Carrera de barras — Top 10 departamentos por año (presiona ▶)',
    labels={'cantidad_m': 'Cantidad (millones)', 'departamento': 'Departamento', 'anio_str': 'Año'},
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_layout(
    height=480,
    showlegend=False,
    yaxis={'categoryorder': 'total ascending'},
)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400
fig.show()


## Parte 12 · Scatter: eventos vs. cantidad promedio por municipio

¿Los municipios con más operativos también incautan más por evento? Detecta outliers y patrones.

In [ ]:
scatter_df = (
    df.groupby(['municipio', 'departamento'])
    .agg(
        total_eventos=('cantidad', 'count'),
        cantidad_promedio=('cantidad', 'mean'),
        cantidad_total=('cantidad', 'sum'),
    )
    .reset_index()
)
scatter_df = scatter_df[scatter_df['total_eventos'] >= 50]  # solo municipios con historia

fig = px.scatter(
    scatter_df,
    x='total_eventos',
    y='cantidad_promedio',
    size='cantidad_total',
    color='departamento',
    hover_name='municipio',
    log_x=True,
    log_y=True,
    title='Municipios: nº de eventos vs. cantidad promedio por evento (tamaño = total incautado)',
    labels={
        'total_eventos':    'Número de eventos (log)',
        'cantidad_promedio':'Cantidad promedio por evento (log)',
        'departamento':     'Departamento',
    },
    opacity=0.7,
)
fig.update_layout(height=520, legend_title='Departamento')
fig.show()


## Parte 13 · Dashboard compacto: subplots en una sola figura

Vista ejecutiva con 4 gráficos en una cuadrícula.

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Cantidad total por sustancia',
        'Eventos por año',
        'Top 10 departamentos',
        'Proporción por sustancia (%)'
    ],
    specs=[[{'type':'xy'},{'type':'xy'}],
           [{'type':'xy'},{'type':'domain'}]],
)

# 1. Barras por sustancia
sust_tot = df.groupby('sustancia')['cantidad'].sum().reset_index().sort_values('cantidad')
fig.add_trace(
    go.Bar(x=sust_tot['cantidad']/1e6, y=sust_tot['sustancia'],
           orientation='h', marker_color=px.colors.qualitative.Bold[:5],
           showlegend=False,
           hovertemplate='%{y}: %{x:.1f}M<extra></extra>'),
    row=1, col=1
)

# 2. Línea de eventos por año
evt_yr = df.groupby('anio').size().reset_index(name='eventos')
fig.add_trace(
    go.Scatter(x=evt_yr['anio'], y=evt_yr['eventos'],
               mode='lines+markers', line_color='#2980b9',
               fill='tozeroy', fillcolor='rgba(41,128,185,0.15)',
               showlegend=False,
               hovertemplate='%{x}: %{y:,} eventos<extra></extra>'),
    row=1, col=2
)

# 3. Top 10 deptos
depto_tot = df.groupby('departamento')['cantidad'].sum().reset_index().sort_values('cantidad').tail(10)
fig.add_trace(
    go.Bar(x=depto_tot['cantidad']/1e6, y=depto_tot['departamento'],
           orientation='h', marker_color='#e74c3c',
           showlegend=False,
           hovertemplate='%{y}: %{x:.1f}M<extra></extra>'),
    row=2, col=1
)

# 4. Pie por sustancia
fig.add_trace(
    go.Pie(labels=sust_tot['sustancia'], values=sust_tot['cantidad'],
           hole=0.4,
           marker_colors=px.colors.qualitative.Bold[:5],
           showlegend=True,
           hovertemplate='%{label}: %{percent}<extra></extra>'),
    row=2, col=2
)

fig.update_layout(
    height=600,
    title_text='📊 Vista ejecutiva — Incautación de Estupefacientes Colombia',
    title_font_size=15,
    paper_bgcolor='white',
    margin=dict(t=80, b=30, l=10, r=10),
)
fig.show()


## Cierre analítico

Responde las siguientes preguntas con base en los gráficos generados:

**1. ¿Qué filtros priorizarías en una app interactiva?**  
*(Ej: año, sustancia, departamento)*

**2. ¿Qué KPIs pondrías en la cabecera del dashboard?**  
*(Ej: total incautado, crecimiento YoY, departamento pico)*

**3. ¿Qué patrones o anomalías detectaste?**  
*(Ej: caída 2019–2020, dominancia de marihuana, outliers en Valle)*

**4. ¿Qué gráfico entrega el insight más rápido y cuál requiere más exploración?**


✍️ *Escribe aquí tu reflexión final.*
